In [17]:
import numpy as np
import pandas as pd
import glob
from tqdm import tqdm
import pickle
import os
import json

In [18]:
train_df = pd.read_csv("D:/LLM/LAB 1/Dataset/train_combined_ngram.csv")
val_df   = pd.read_csv("D:/LLM/LAB 1/Dataset/val_combined_ngram.csv")
test_df  = pd.read_csv("D:/LLM/LAB 1/Dataset/test_combined_ngram.csv")
full_df = pd.read_csv("D:/LLM/LAB 1/Dataset/full_data_ngram.csv")


print(f"Full data size: {len(full_df)} Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Full data size: 24000 Train size: 19200, Val size: 2400, Test size: 2400


Build vocabulary from training data

In [19]:
def tokenize(sentence):
    return str(sentence).split()

Calculate TF (Term Frequency)

In [20]:
def tf(sentence, word_to_idx):

    tf_value={}

    tokens=tokenize(sentence)

    for token in tokens:
        idx = word_to_idx.get(token)

        if idx is not None:
            tf_value[idx]=tf_value.get(idx,0)+1

    return tf_value

Calculate IDF (Inverse Document Frequency)

In [21]:
def idf(train_df,word_to_idx):
    N = len(train_df)  # Total documents
    df_value=np.zeros(len(word_to_idx),dtype=np.uint16)

    for sentence in tqdm(train_df,total=N,desc="Computing IDF"):
        tokens=set(tokenize(sentence))

        for t in tokens:
            idx=word_to_idx.get(t)
            if idx is not None:
                df_value[idx]=df_value[idx]+1

    idf_values=np.log((N+1)/(df_value+1))+1

    return idf_values

Calculate TF-IDF for each document

In [22]:
def calculate_tfidf(sentences, idf_values, vocab, word_to_idx):

    tfidf_matrix = {}
    
    for doc_idx, sentence in tqdm(enumerate(sentences), total=len(sentences), desc="Computing TF-IDF"):
        
        # Calculate term frequency
        tf_values = tf(sentence,word_to_idx)
        
        result={}
        for idx,tf_v in tf_values.items():
            temp=tf_v*idf_values[idx]

            if temp != 0:
                result[idx]=temp

        tfidf_matrix[doc_idx]=result
    
    return tfidf_matrix

implementation

In [23]:
full_df["text"].shape


(24000,)

# impimantation 

In [24]:
to_work_on = [train_df, val_df, test_df]

vocab_folder = r"D:/LLM/LAB 1/Dataset/vocab"
tfidf_folder = r"D:/LLM/LAB 1/Dataset/tfidf"

pkl_files = glob.glob(
    os.path.join(vocab_folder, "*.pkl")
)

print(f"Found {len(pkl_files)} pickle files")

Found 36 pickle files


In [25]:
for pkl_path in pkl_files:

    pkl_name = os.path.splitext(
        os.path.basename(pkl_path)
    )[0]

    
    print(f"Processing: {pkl_name}")
    print("=" * 80)


    output_folder = os.path.join(tfidf_folder,pkl_name)
    os.makedirs(output_folder,exist_ok=True)

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    all_words = ( word for lang_vocab in data.values() for word in lang_vocab)
    unique_words = set(all_words)
    word_to_idx = { word: idx for idx, word in enumerate(unique_words) }
    vocab = word_to_idx.keys()

    idf_values=idf(full_df["text"],word_to_idx)

    with open(os.path.join(output_folder, "idf.pkl"), 'wb') as file:
        pickle.dump(idf_values, file)

    
    with open(os.path.join(output_folder, "word_to_idx.pkl"), 'wb') as file:
        pickle.dump(word_to_idx, file)


    output_paths = [
        os.path.join(output_folder, "train_df.csv"),
        os.path.join(output_folder, "val_df.csv"),
        os.path.join(output_folder, "test_df.csv")
    ]


    for df, output_path in zip(to_work_on,output_paths):

        X = calculate_tfidf(
            df["text"],
            idf_values,
            vocab,
            word_to_idx
        )

        new_df = df.copy()
        new_df["text"] = [ json.dumps(vector) for vector in X.values()]
        new_df.to_csv(output_path,index=False)

        del new_df

        print(f"Saved: {output_path}")

    del data
    del idf_values
    del vocab
    del word_to_idx

    print(f"Finished: {pkl_name}")


Processing: between_01_02


Computing IDF:   0%|          | 0/24000 [00:00<?, ?it/s]

Computing TF-IDF: 100%|██████████| 19200/19200 [00:01<00:00, 16987.72it/s]


Saved: D:/LLM/LAB 1/Dataset/tfidf\between_01_02\train_df.csv


Computing TF-IDF: 100%|██████████| 2400/2400 [00:00<00:00, 13318.67it/s]


...

Saved: D:/LLM/LAB 1/Dataset/tfidf\top_09\train_df.csv


Computing TF-IDF: 100%|██████████| 2400/2400 [00:00<00:00, 6927.28it/s]


Saved: D:/LLM/LAB 1/Dataset/tfidf\top_09\val_df.csv


Computing TF-IDF: 100%|██████████| 2400/2400 [00:00<00:00, 6253.24it/s]


Saved: D:/LLM/LAB 1/Dataset/tfidf\top_09\test_df.csv
Finished: top_09
